# Train YOLOv8 on Retail Products Classification

GPU half of the VS Code workflow. You edit the repo locally, push, and this notebook pulls
that commit and trains on Colab's GPU. Nothing is edited here — cell 3 hard-resets to origin.

**Run order:** 1 → 8 top to bottom. Cell 7 is a cheap smoke run; only go to cell 8 once it passes.

The dataset labels whole images across 21 categories, so this trains `yolov8n-cls`
(classification). Pass `--task detect` instead to train the full-frame-box detector shim.

## 1. Check the GPU
If this fails: **Runtime → Change runtime type → T4 GPU**, then re-run.

In [ ]:
import subprocess

import torch

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip() or "nvidia-smi returned nothing")

assert torch.cuda.is_available(), "No GPU attached. Runtime > Change runtime type > T4 GPU."
print(f"torch {torch.__version__} | cuda {torch.version.cuda}")

## 2. Mount Drive
Colab wipes `/content` when the runtime recycles, so weights and the Kaggle token live in Drive.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/yolo-retail"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("artifacts will be saved to", DRIVE_DIR)

## 3. Pull the repo

This is the bridge: whatever you last pushed from VS Code is what trains here.

`git reset --hard` **discards any edit made inside Colab** — that is deliberate, so the
notebook can never train a version that does not exist in git. Edit in VS Code, push, re-run.

Private repo? Swap `REPO_URL` for `https://<token>@github.com/weshallsah/yolo.git` using a
fine-grained PAT with read-only Contents access.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/weshallsah/yolo.git"
BRANCH = "main"
REPO_DIR = "/content/yolo"


def sh(*args, cwd=None):
    print("$", " ".join(args))
    subprocess.run(args, cwd=cwd, check=True)


if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "fetch", "origin", BRANCH, cwd=REPO_DIR)
    sh("git", "checkout", BRANCH, cwd=REPO_DIR)
    sh("git", "reset", "--hard", f"origin/{BRANCH}", cwd=REPO_DIR)
else:
    sh("git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR)

os.chdir(REPO_DIR)
sh("git", "log", "--oneline", "-1")

## 4. Install dependencies
Only what training needs — not the FastAPI serving stack.

In [ ]:
!pip install -q "ultralytics==8.4.150" kaggle

import ultralytics

ultralytics.checks()

## 5. Kaggle credentials

Get a token at **kaggle.com → Settings → API → Create New Token**. It downloads `kaggle.json`.

Easiest path: put that file in Drive at `MyDrive/kaggle/kaggle.json` once, and every future
runtime picks it up with no prompt. Otherwise this cell falls back to an upload widget.

The token is a credential — it is copied, never printed.

In [ ]:
import json
import os
import shutil

DRIVE_TOKEN = "/content/drive/MyDrive/kaggle/kaggle.json"
TARGETS = ["/root/.kaggle/kaggle.json", "/root/.config/kaggle/kaggle.json"]

if os.path.exists(DRIVE_TOKEN):
    source = DRIVE_TOKEN
    print("using the token already in Drive")
else:
    from google.colab import files

    print("No token at", DRIVE_TOKEN, "- upload kaggle.json (Kaggle > Settings > Create New Token)")
    source = next(iter(files.upload()))

# Written to both paths because which one the CLI reads depends on its version.
for target in TARGETS:
    os.makedirs(os.path.dirname(target), exist_ok=True)
    shutil.copy(source, target)
    os.chmod(target, 0o600)

with open(source) as handle:
    print("credentials installed for Kaggle user:", json.load(handle).get("username"))

## 6. Download the competition data

**You must accept the rules first**, or the API returns 403:
<https://www.kaggle.com/c/retail-products-classification/rules> → *I Understand and Accept*.

Archives are unpacked into `/content/retail_data`, which is one of the roots
`prepare_retail_dataset.py` searches, so no `--csv` flag is needed afterwards.

In [ ]:
import glob
import os
import subprocess
import zipfile

COMPETITION = "retail-products-classification"
DATA_DIR = "/content/retail_data"
os.makedirs(DATA_DIR, exist_ok=True)

result = subprocess.run(
    ["kaggle", "competitions", "download", "-c", COMPETITION, "-p", DATA_DIR],
    capture_output=True, text=True,
)
print(result.stdout, result.stderr)
if result.returncode != 0:
    raise SystemExit(
        "Download failed. The usual cause is unaccepted competition rules: open "
        f"https://www.kaggle.com/c/{COMPETITION}/rules, click 'I Understand and Accept', "
        "then re-run this cell."
    )

# Two passes: the outer archive, then any zips nested inside it (train.csv.zip, image zips).
# Each archive is deleted once extracted, since Colab's disk is smaller than you think.
for _ in range(2):
    for archive in glob.glob(os.path.join(DATA_DIR, "**", "*.zip"), recursive=True):
        print("extracting", archive)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(os.path.dirname(archive))
        os.remove(archive)

!du -sh {DATA_DIR} && find {DATA_DIR} -maxdepth 2 | head -20

## 7. Smoke run
Builds a 2,000-image subset and trains one epoch. Catches a bad download, a column-name
surprise or an OOM in ~2 minutes instead of an hour into the real run.

In [ ]:
!python backend/scripts/train_on_retail.py \
    --task classify \
    --max-images 2000 \
    --epochs 1 \
    --batch 64 \
    --workers 2 \
    --cache none

## 8. Full training run

`--force` is required: cell 7 left a capped 2,000-image dataset behind, and without it the
build is skipped and you would train on the subset again.

`--cache disk` because ~42k images will not fit in Colab's RAM. Watch `top-1` accuracy.
Raise `--epochs` if it is still climbing when it stops; add `--max-per-class 3000` if the
per-class counts printed by the prep step look badly skewed.

In [ ]:
!python backend/scripts/train_on_retail.py \
    --task classify \
    --force \
    --epochs 30 \
    --imgsz 224 \
    --batch 64 \
    --workers 2 \
    --cache disk

## 9. Save the run to Drive
Do this before the runtime recycles, or the weights are gone.

In [ ]:
import os
import shutil

RUN_DIR = "backend/models/retail_yolo_products_cls"
destination = os.path.join(DRIVE_DIR, "retail_yolo_products_cls")

shutil.copytree(RUN_DIR, destination, dirs_exist_ok=True)
print("saved to", destination)
!ls -la {destination}/weights

## 10. Sanity-check the trained classifier
Predicts a few held-out validation images. The printed class should usually match the
directory the image came from.

In [ ]:
import glob
import os

from ultralytics import YOLO

model = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))
samples = sorted(glob.glob("backend/training_data/retail/dataset_products_cls/val/*/*"))[:10]

for result in model.predict(samples, verbose=False):
    truth = os.path.basename(os.path.dirname(result.path))
    predicted = result.names[result.probs.top1]
    mark = "ok " if truth == predicted else "MISS"
    print(f"{mark} true={truth:28s} pred={predicted:28s} conf={result.probs.top1conf:.2f}")

## Getting the weights back into VS Code

The run directory is in Drive, so download `best.pt` from
`MyDrive/yolo-retail/retail_yolo_products_cls/weights/` and drop it in the repo (it is
gitignored — `backend/models/` is not tracked), then point the backend at it:

```bash
# backend/.env
APP_YOLO_WEIGHTS_PATH=models/retail_yolo_products_cls/weights/best.pt
APP_MODEL_TASK=classify
```

`APP_MODEL_TASK=classify` is required. The default serving path reads `result.boxes`, which a
classifier does not produce — without it the API returns an empty list for every image.